# Featured Quotes for Artists

Get one random sample for `source_actor == "artist"` point per artist from `artist_points_flat.json` and write the result to a csv - will overwrite later based on manual curation

In [13]:
import json
import random
from pathlib import Path

import pandas as pd

In [14]:
artist_points_path = Path("../../data/clusters/artist_points_flat.json")
output_path = Path("../../data/addl/featured_quotes.csv")

with artist_points_path.open(encoding="utf-8") as f:
    artist_points = json.load(f)

artist_points_by_artist = {
    str(entry["artist_id"]): entry["points"]
    for entry in artist_points
}

In [15]:
rng = random.Random(42)

rows = []
for artist_id in sorted(artist_points_by_artist, key=lambda value: int(value)):
    points = [
        point
        for point in artist_points_by_artist.get(artist_id, [])
        if point.get("source_actor") == "artist" and not point.get("is_artwork")
    ]
    if not points:
        rows.append(
            {
                "artist_id": artist_id,
                "source_idx": pd.NA,
                "point": pd.NA,
                "text": pd.NA,
            }
        )
        continue

    selected = rng.choice(points)
    rows.append(
        {
            "artist_id": artist_id,
            "source_idx": selected.get("source_idx", pd.NA),
            "point": selected.get("id", pd.NA),
            "text": selected.get("text", pd.NA),
        }
    )

featured_quotes = pd.DataFrame(rows, columns=["artist_id", "source_idx", "point", "text"])

missing_points = int(featured_quotes["point"].isna().sum())
if missing_points:
    print(f"did not find featured quotes for {missing_points} artists")

featured_quotes[["artist_id", "source_idx", "point"]].to_csv(output_path, index=False)
display(featured_quotes.head(20))

,artist_id,source_idx,point,text
0,0,AT56,2,"… You got to take my integrity, you know, as m..."
1,1,AT01,4,Navigating this stuff allows me to investigate...
2,2,AT10,7,"In my early videos, I physically appeared in t..."
3,3,AT23,12,Largely related to my unusual ethnic heritage ...
4,4,AT25,27,"On a larger scope, the project has helped push..."
5,5,AT36,34,I don’t think of myself as Japanese. I think o...
6,6,AT43,40,I’m a transnational artist and I’ve benefited ...
7,7,AT58,42,"For me, language was just something I’ve alway..."
8,8,AT68,53,"I think that’s where I discovered my language,..."
9,9,AT71,62,"Through making the documentary Wildness, I had..."
